# Publish Model to HF Hub

Select a model checkpoint from W&B artifacts and publish it as the official
`kaya-go/moku-v2` model on Hugging Face Hub.

**Workflow:**

1. List available W&B model artifacts (with mAP, epoch, run info)
2. Set the artifact path to publish
3. Download and load the model
4. Quick sanity check on real test set
5. Push to HF Hub

## Available Artifacts

Browse model artifacts from W&B and pick the one to publish.

In [9]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv

load_dotenv()

from datasets import load_dataset

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.evaluation import evaluate_map, format_map_per_class, format_map_results
from moku.runs import list_wandb_model_artifacts, load_model_from_wandb
from moku.model import make_eval_transform

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Select Artifact

Set the artifact path below. Use the `name` column from the table above.
Append `:latest` or `:vN` for a specific version.

In [6]:
# ← Set the artifact to publish
ARTIFACT = "model-r5_lr4e-4_linear500:latest"

HF_MODEL = "kaya-go/moku-v2"
HF_DATASET = "kaya-go/moku-v2"

print(f"Will publish: {ARTIFACT}")
print(f"Target: {HF_MODEL}")

Will publish: model-r5_lr4e-4_linear500:latest
Target: kaya-go/moku-v2


## Download & Load Model

In [7]:
ip, model = load_model_from_wandb(ARTIFACT)

total = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total:,} parameters")
print(f"Labels: {model.config.id2label}")

wandb: Downloading large artifact 'model-r5_lr4e-4_linear500:latest', 76.78MB. 3 files...
wandb:   3 of 3 files downloaded.  
Done. 00:00:00.6 (130.5MB/s)


Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

Model loaded: 20,075,740 parameters
Labels: {0: 'black_stone', 1: 'white_stone', 2: 'board_corner'}


## Sanity Check — mAP on Real Test

Quick evaluation on the real test split to confirm the model performs as expected.

In [10]:
ds_real = load_dataset(HF_DATASET, "real")
ds_real.set_transform(make_eval_transform(ip))

metrics = evaluate_map(
    model=model,
    dataset=ds_real["test"],
    image_processor=ip,
    batch_size=8,
)

print("=== Overall mAP ===")
display(format_map_results(metrics))

print("\n=== Per-Class AP ===")
display(format_map_per_class(metrics))

=== Overall mAP ===


,metric,value
0,mAP@50:95,0.3697
1,mAP@50,0.7854
2,mAP@75,0.2718
3,mAR@400,0.5121



=== Per-Class AP ===


,category,AP@50:95
0,black_stone,0.3399
1,white_stone,0.4183
2,board_corner,0.3509


## Push to HF Hub

Publish the model and image processor to the official HF model repository.

**Warning**: This overwrites the current model on the `main` branch of `kaya-go/moku-v2`.

In [11]:
# Uncomment to push (destructive — overwrites current model on HF Hub)
model.push_to_hub(HF_MODEL)
ip.push_to_hub(HF_MODEL)
print(f"Model pushed to https://huggingface.co/{HF_MODEL}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Model pushed to https://huggingface.co/kaya-go/moku-v2
